# Interactive Flight Planning

This notebook demonstrates hyplan's modular interactive planning widgets.

**Requirements:** `pip install hyplan[gui]`

## Widget overview

| Widget | Purpose |
|--------|---------|
| **FlightPlanner** | Full integrated planner (composes all widgets below) |
| **WaypointEditor** | Table + draggable map markers |
| **FlightBoxWidget** | Draw/reshape polygon, generate flight lines |
| **PatternWidget** | Place/drag center point, generate patterns with sliders |
| **FlightPlanViewer** | Compute flight plan + altitude profile |
| **LayerPanel** | Toggle basemaps, FAA charts, weather, satellite layers |
| **AirspaceLayer** | OpenAIP airspace polygons + conflict detection |
| **ExportPanel** | Export flight plan in multiple formats |

Each widget can be used **standalone** or **composed together** on a shared map.

## 1. Setup

In [ ]:
from hyplan import (
    DynamicAviation_B200, NASA_ER2,
    AVIRIS3, ureg,
    FlightLine, Waypoint,
    racetrack, rosette, polygon, sawtooth, spiral,
    box_around_center_line,
)
from hyplan.airports import Airport, initialize_data

# New modular widget imports
from hyplan.gui import (
    PlannerState,
    FlightPlanner,
    WaypointEditor,
    FlightBoxWidget,
    PatternWidget,
    FlightPlanViewer,
    LayerPanel,
    AirspaceLayer,
    ExportPanel,
)
import ipyleaflet
import ipywidgets as widgets

# Initialize airport data
initialize_data(countries=["US"])

# Aircraft and sensor
b200 = DynamicAviation_B200()
sensor = AVIRIS3()
kedw = Airport("KEDW")  # Edwards AFB

CENTER = (34.9, -117.9)
ALT = ureg.Quantity(20000, "feet")

print(f"Aircraft: {b200}")
print(f"Sensor:   {sensor}")
print(f"Airport:  {kedw.name} ({kedw.icao_code})")

## 2. Full Integrated Planner

The `FlightPlanner` composes all widgets into a single interface — backward compatible with the original API.

**Controls:**
- **Place WP**: Click map to add waypoints
- **Pan**: Navigate without placing waypoints
- **Tabs**: Switch between Patterns, Flight Box, Layers, Airspace, and Export panels

In [ ]:
planner = FlightPlanner(
    aircraft=b200,
    takeoff_airport=kedw,
    return_airport=kedw,
    instrument=sensor,
    center=CENTER,
    zoom=9,
)
planner.show()

After placing waypoints and clicking Compute, retrieve the results:

In [ ]:
# Access the computed flight plan
if planner.flight_plan is not None:
    print(f"Waypoints: {len(planner.waypoints)}")
    plan = planner.flight_plan
    total_dist = plan['distance'].dropna().sum()
    total_time = plan['time_to_segment'].sum()
    print(f"Total distance: {total_dist:.1f} nm")
    print(f"Total time: {total_time:.1f} min ({total_time/60:.1f} hr)")
    display(plan[['segment_type', 'segment_name', 'distance', 'time_to_segment']].head(10))
else:
    print("No flight plan computed yet. Click 'Compute' in the widget above.")

## 3. Standalone Widgets on a Shared Map

Each widget can be used independently. Here we create a shared map and attach multiple widgets to it. All widgets share the same `PlannerState` — changes made in one are visible to the others.

In [ ]:
# Create shared state and map
state = PlannerState(
    aircraft=b200,
    takeoff_airport=kedw,
    return_airport=kedw,
    instrument=sensor,
)

m = ipyleaflet.Map(
    center=CENTER,
    zoom=10,
    scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="500px"),
)

# Attach widgets to the shared map
wp_editor = WaypointEditor(state, m)
box_widget = FlightBoxWidget(state, m)
pattern_widget = PatternWidget(state, m)
plan_viewer = FlightPlanViewer(state, m)
layers = LayerPanel(m)
export = ExportPanel(state)

# Layout: map on top, widgets in tabs below
tabs = widgets.Tab(children=[
    wp_editor,
    box_widget,
    pattern_widget,
    layers,
    plan_viewer,
    export,
])
tabs.set_title(0, "Waypoints")
tabs.set_title(1, "Flight Box")
tabs.set_title(2, "Patterns")
tabs.set_title(3, "Layers")
tabs.set_title(4, "Flight Plan")
tabs.set_title(5, "Export")

display(widgets.VBox([m, tabs]))

## 4. FlightBoxWidget — Editable Polygon

Draw a polygon on the map, then **drag vertices** to reshape it. Flight lines auto-regenerate on vertex drag or slider change.

1. Draw a polygon or rectangle on the map
2. Drag the orange vertex handles to reshape
3. Adjust azimuth, overlap, altitude sliders — lines update live
4. Click **Generate Flight Box** to create flight lines

In [ ]:
# FlightBoxWidget standalone demo
box_state = PlannerState(instrument=sensor)
box_map = ipyleaflet.Map(
    center=CENTER, zoom=10, scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="450px"),
)
box_demo = FlightBoxWidget(box_state, box_map)
LayerPanel(box_map)  # attach layer controls to same map

display(widgets.HBox([
    box_map,
    widgets.VBox([box_demo], layout=widgets.Layout(width="320px")),
]))

## 5. PatternWidget — Draggable Center + Sliders

Click the map to place a pattern center, pick a pattern type, and adjust parameters with sliders. Drag the center marker to reposition — the pattern regenerates automatically.

Supported patterns: **Racetrack**, **Rosette**, **Sawtooth**, **Spiral**, **Polygon**

In [ ]:
# PatternWidget standalone demo
pat_state = PlannerState()
pat_map = ipyleaflet.Map(
    center=CENTER, zoom=10, scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="450px"),
)
pat_demo = PatternWidget(pat_state, pat_map)

display(widgets.HBox([
    pat_map,
    widgets.VBox([pat_demo], layout=widgets.Layout(width="320px")),
]))

## 6. LayerPanel — Background Layers

Toggle basemaps, FAA aeronautical charts, GOES/NEXRAD weather, and NASA GIBS satellite imagery. The GIBS date picker controls the date for satellite layers.

In [ ]:
# LayerPanel standalone demo
layer_map = ipyleaflet.Map(
    center=CENTER, zoom=8, scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="450px"),
)
layer_demo = LayerPanel(layer_map)

display(widgets.HBox([
    layer_map,
    widgets.VBox([layer_demo], layout=widgets.Layout(width="280px")),
]))

## 7. Loading Pre-existing Data

Pass waypoints or flight lines created in code to any widget via `PlannerState`.

In [ ]:
# Build a complex flight sequence in code
wps_combined = []

# Racetrack survey
wps_combined.extend(racetrack(
    (35.0, -117.5), 90.0, ALT,
    ureg.Quantity(40, "km"), n_legs=4, offset=ureg.Quantity(5, "km"),
))

# Spiral descent
wps_combined.extend(spiral(
    (35.0, -118.3), 0.0,
    altitude_start=ALT,
    altitude_end=ureg.Quantity(5000, "feet"),
    radius=ureg.Quantity(5, "km"),
    n_turns=2,
))

print(f"Combined sequence: {len(wps_combined)} waypoints")

# Load into the full planner for interactive editing
planner_pre = FlightPlanner(
    aircraft=b200,
    takeoff_airport=kedw,
    return_airport=kedw,
    waypoints=wps_combined,
    instrument=sensor,
    center=(35.0, -117.9),
    zoom=9,
)
planner_pre.show()

## 8. Programmatic Access

The `PlannerState` is observable — you can read and write state from code, and widgets update automatically.

In [ ]:
# Read state from any widget
print(f"Waypoints in shared state: {len(state.waypoints)}")
print(f"Flight lines in shared state: {len(state.flight_lines)}")
print(f"Aircraft: {state.aircraft}")

# Add a waypoint programmatically — the WaypointEditor updates automatically
state.append_waypoint(Waypoint(
    latitude=34.9, longitude=-117.8,
    heading=0.0, altitude_msl=ALT,
    name="Programmatic_WP",
))
print(f"\nAfter adding: {len(state.waypoints)} waypoints")

# The full planner's state is also accessible
print(f"\nFull planner waypoints: {len(planner.state.waypoints)}")
print(f"Full planner flight lines: {len(planner.state.flight_lines)}")